# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

- Se importan las librerías necesarias
- Se cargan los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Se guardan los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Y se explora cada dataset.
---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [ ]:
# explorar datasets
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
catalog.info()
catalog.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# Copia de DataFrame original
orders_clean = orders.copy()

# Conversión de fecha a datetime
orders_clean['fecha_hora_pedido'] = pd.to_datetime(orders_clean['fecha_hora_pedido'], errors='coerce')

# Limpieza de IDs quitando espacios
orders_clean['id_pedido'] = orders_clean['id_pedido'].astype(str).str.strip()
orders_clean['id_usuario'] = orders_clean['id_usuario'].astype(str).str.strip()

# Convesión de columnas numéricas
numeric_cols = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
for col in numeric_cols:
    orders_clean[col] = pd.to_numeric(orders_clean[col], errors='coerce')

# Eliminando duplicados
orders_clean.drop_duplicates(inplace = True)

# monto_descuento puede ser 0, pero no negativo
orders_clean = orders_clean[orders_clean['monto_descuento'] >= 0]

# Eliminación de ceros inválidos
orders_clean = orders_clean[orders_clean['precio_unitario']!= 0]
orders_clean = orders_clean[orders_clean['cantidad']!= 0]

# Eliminación de valores negativos inválidos y/o imposibles
numeric_cols = ['cantidad', 'precio_unitario', 'monto_total']
for col in numeric_cols:
    orders_clean = orders_clean[orders_clean[col]>=0]
    orders_clean = orders_clean[orders_clean[col] <= 1000]
    orders_clean = orders_clean.dropna(subset=[col])

# Detectando outliers
Q1 = orders_clean['cantidad'].quantile(0.25)
Q3 = orders_clean['cantidad'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

orders_clean = orders_clean[(orders_clean['cantidad'] >= lower) & (orders_clean['cantidad'] <= upper)]

# Validando consistencia de montos
orders_clean['monto_calculado'] = orders_clean['cantidad'] * orders_clean['precio_unitario'] - orders_clean['monto_descuento']

# Tolerancia por decimales
orders_clean = orders_clean[np.isclose(orders_clean['monto_total'], orders_clean['monto_calculado'], atol = 0.01)]
orders_clean.drop(columns = ['monto_calculado'], inplace = True)

# id_pedido debe ser único
orders_clean = orders_clean.drop_duplicates(subset = 'id_pedido', keep = 'first')

# Limpiando categóricas
cat_cols = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']

for col in cat_cols:
    orders_clean[col] = orders_clean[col].astype(str).str.strip().str.lower()

orders_clean.to_csv("orders_clean.csv", index=False)

orders_clean.info()
orders_clean.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24936 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24936 non-null  object        
 1   id_usuario          24936 non-null  object        
 2   fecha_hora_pedido   24936 non-null  datetime64[ns]
 3   pais                24936 non-null  object        
 4   dispositivo         24936 non-null  object        
 5   fuente_referencia   24936 non-null  object        
 6   nombre_producto     24936 non-null  object        
 7   categoria_producto  24936 non-null  object        
 8   cantidad            24936 non-null  float64       
 9   precio_unitario     24936 non-null  float64       
 10  monto_descuento     24936 non-null  float64       
 11  monto_total         24936 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,argentina,desktop,organic,jacket-winter-m,moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,mexico,desktop,paid_search,tablet-standard-64gb,electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,argentina,desktop,social,blender-xl-red,hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,colombia,mobile,social,tablet-standard-64gb,electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,argentina,desktop,paid_search,blender-xl-red,hogar,1.0,336.28,0.0,336.28


In [ ]:
# Copia de DataFrame original
catalog_clean = catalog.copy()

# Limpieza de columnas categóricas
cat_cols = ['nombre_producto', 'categoria_producto', 'proveedor']

for col in cat_cols:
    catalog_clean[col] = (
        catalog_clean[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

# Convirtiendo numéricas
catalog_clean['costo_unitario'] = pd.to_numeric(catalog_clean['costo_unitario'], errors = 'coerce')

# Validando valores numéricos
# No negativos
catalog_clean = catalog_clean[catalog_clean['costo_unitario'] >= 0]
# No ceros inválidos
catalog_clean = catalog_clean[catalog_clean['costo_unitario'] != 0]

# Eliminando duplicados
catalog_clean.drop_duplicates(inplace = True)

catalog_clean.info()
catalog_clean.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 280.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,laptop-gaming-16gb,electrónica,280.68,"fuller, pena and myers"
1,phone-pro-128gb,electrónica,10.12,king ltd
2,tablet-standard-64gb,electrónica,25.21,bowers llc
3,blender-xl-red,hogar,176.64,long-reid
4,vacuum-pro-black,hogar,16.60,"rivera, carr and finley"


In [ ]:
# Copia de DataFrame original
marketing_clean = marketing.copy()

# Convirtiendo fecha a datetime
marketing_clean['fecha'] = pd.to_datetime(marketing_clean['fecha'], errors = 'coerce')

# Limpiando columnas categóricas
cat_cols = ['pais', 'id_campaña', 'canal']
for col in cat_cols:
    marketing_clean[col] = (
        marketing_clean[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

# Manejando nulos
# canal tiene nulos, reemplazamos por "desconocido"
marketing_clean['canal'] = marketing_clean['canal'].replace("nan", np.nan)
marketing_clean['canal'] = marketing_clean['canal'].fillna("desconocido")

# Convirtiendo columna numérica
marketing_clean['gasto'] = pd.to_numeric(marketing_clean['gasto'], errors = 'coerce')

# Validando valores numéricos
# No negativos
marketing_clean = marketing_clean[marketing_clean['gasto'] >= 0]

# No ceros inválidos
marketing_clean = marketing_clean[marketing_clean['gasto'] != 0]

# Eliminando duplicados
marketing_clean.drop_duplicates(inplace = True)

marketing_clean.info()
marketing_clean.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 75.9+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,mexico,organic_mexico,organic,2446.25
1,2025-01-01,mexico,paid_search_mexico,paid_search,2704.34
2,2025-01-01,mexico,social_mexico,social,2045.01
3,2025-01-01,colombia,organic_colombia,organic,2597.21
4,2025-01-01,colombia,paid_search_colombia,paid_search,1771.40


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# PARTE 1: RENTABILIDAD DEL NEGOCIO

# 1. Revenue total
revenue_total = orders_clean['monto_total'].sum()

# 2. Costo total
# Se une orders con catalog para obtener costo_unitario
orders_cost = orders_clean.merge(
    catalog_clean[['nombre_producto', 'costo_unitario']],
    on = 'nombre_producto',
    how = 'left'
)
# Costo total = cantidad * costo_unitario
orders_cost['costo_total'] = orders_cost['cantidad'] * orders_cost['costo_unitario']
costo_total = orders_cost['costo_total'].sum()

# 3. Gasto total en marketing
gasto_marketing_total = marketing_clean['gasto'].sum()

# 4. Profit
profit = revenue_total - costo_total - gasto_marketing_total

# 5. Margen de utilidad
profit_margin = (profit / revenue_total) * 100

# Mostrar resultados
kpis = pd.DataFrame({
    'Métrica': ['Revenue total', 'Costo total', 'Gasto marketing', 'Profit', 'Margen (%)'],
    'Valor': [revenue_total, costo_total, gasto_marketing_total, profit, profit_margin]
})
pd.set_option('display.float_format', '{:,.0f}'.format)
kpis

,Métrica,Valor
0,Revenue total,"9,622,282"
1,Costo total,"3,828,869"
2,Gasto marketing,"2,871,844"
3,Profit,"2,921,569"
4,Margen (%),30


In [ ]:
# PARTE 2: COMPORTAMIENTO DE VENTAS

# 1. Ticket promedio por orden
ticket_promedio = orders_clean['monto_total'].mean()
ticket_promedio

# 2. Cantidad promedio de productos por orden
cantidad_promedio = orders_clean['cantidad'].mean()
cantidad_promedio

# 3. Producto más vendido
producto_mas_vendido = (
    orders_clean.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)
producto_mas_vendido

# 4. Gasto en marketing por canal
gasto_por_canal = (
    marketing_clean.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)
gasto_por_canal

resumen_parte2 = pd.DataFrame({
    'Métrica': [
        'Ticket promedio',
        'Cantidad promedio por orden',
        'Producto más vendido'
    ],
    'Valor': [
        ticket_promedio,
        cantidad_promedio,
        producto_mas_vendido.index[0]
    ]
})
resumen_parte2


,Métrica,Valor
0,Ticket promedio,386
1,Cantidad promedio por orden,2
2,Producto más vendido,vacuum-pro-black


In [ ]:
tabla_gasto_canal = gasto_por_canal.reset_index()
tabla_gasto_canal.columns = ['Canal', 'Gasto']
tabla_gasto_canal

,Canal,Gasto
0,social,"918,043"
1,organic,"913,533"
2,paid_search,"863,088"
3,desconocido,"177,179"


# ---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos,
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
    END AS orden
FROM events
GROUP BY nombre_evento
ORDER BY orden;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos,orden
0,first_visit,7796,1
1,select_item,7582,2
2,add_to_cart,7634,3
3,begin_checkout,7208,4
4,add_payment_info,6250,5
5,purchase,6240,6


In [ ]:
# PARTE 2: Conversiones

query_conversion = '''
WITH eventos_limpios AS (
    SELECT
        id_usuario,
        nombre_evento,
        MIN(timestamp_evento) AS primer_evento
        FROM events
        GROUP BY id_usuario, nombre_evento
),
funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden
    FROM eventos_limpios
    GROUP BY nombre_evento
),
ordenado AS (
    SELECT *
    FROM funnel
    ORDER BY orden
),
conversiones AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (ORDER BY orden) AS usuarios_previos,
        CASE
            WHEN LAG(usuarios_unicos) OVER (ORDER BY orden) IS NULL THEN NULL
            ELSE usuarios_unicos * 1.0 / LAG(usuarios_unicos) OVER (ORDER BY orden)
        END AS tasa_conversion,
        CASE
            WHEN LAG(usuarios_unicos) OVER (ORDER BY orden) IS NULL THEN NULL
            ELSE LAG(usuarios_unicos) OVER (ORDER BY orden) - usuarios_unicos
        END AS dropoff
    FROM ordenado
)
SELECT *
FROM conversiones;
'''
conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,usuarios_previos,tasa_conversion,dropoff
0,first_visit,7796,NaN,NaN,NaN
1,select_item,7582,"7,796",1,214
2,add_to_cart,7634,"7,582",1,-52
3,begin_checkout,7208,"7,634",1,426
4,add_payment_info,6250,"7,208",1,958
5,purchase,6240,"6,250",1,10


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        u.id_usuario,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS cohorte,
        CAST(u.fecha_registro AS DATE) AS fecha_registro
    FROM users u
),
actividad AS (
    SELECT
        ua.id_usuario,
        CAST(ua.fecha_actividad AS DATE) AS fecha_actividad,
        DATE_TRUNC('week', CAST(ua.fecha_actividad AS DATE)) AS semana_actividad
    FROM user_activity ua
),
cohorte_actividad AS (
    SELECT
        c.cohorte,
        c.id_usuario,
        EXTRACT(WEEK FROM a.fecha_actividad) - EXTRACT(WEEK FROM c.fecha_registro) AS semanas_desde_registro
    FROM cohortes c
    LEFT JOIN actividad a
        ON c.id_usuario = a.id_usuario
)
SELECT
    cohorte,
    COUNT(DISTINCT id_usuario) AS usuarios_iniciales,
    COUNT(DISTINCT CASE WHEN semanas_desde_registro = 1 THEN id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN semanas_desde_registro = 2 THEN id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN semanas_desde_registro = 3 THEN id_usuario END) AS retenido_w3,
    COUNT(DISTINCT CASE WHEN semanas_desde_registro = 1 THEN id_usuario END) * 1.0 / COUNT(DISTINCT id_usuario) AS semana_1,
    COUNT(DISTINCT CASE WHEN semanas_desde_registro = 2 THEN id_usuario END) * 1.0 / COUNT(DISTINCT id_usuario) AS semana_2,
    COUNT(DISTINCT CASE WHEN semanas_desde_registro = 3 THEN id_usuario END) * 1.0 / COUNT(DISTINCT id_usuario) AS semana_3
FROM cohorte_actividad
GROUP BY cohorte
ORDER BY cohorte;
 '''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,1627,1627,1627,1,1,1
1,2025-02-01 00:00:00+00:00,1444,1444,1444,1444,1,1,1
2,2025-03-01 00:00:00+00:00,1636,1636,1636,1636,1,1,1
3,2025-04-01 00:00:00+00:00,1606,1606,1606,1606,1,1,1
4,2025-05-01 00:00:00+00:00,1687,1687,1687,1687,1,1,1


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Se analiza el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Se plantea la hipótesis estadística**     
3. **Se aplica el test estadístico adecuado**
4. **Se interpreta el resultado**  

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

#Cargar dataset
df = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")

# Explorar la estructura del dataset
print("Información del dataset:")
print(df.info())
print("\nPrimeras filas:")
print(df.head())
print("\nNombres de las columnas:")
print(df.columns.tolist())

Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB
None

Primeras filas:
   id_usuario     variante  convirtio dispositivo       pais  duracion_sesion  \
0  exp_user_0  tratamiento          0      mobile  Argentina              114   
1  exp_user_1  tratamiento          0     desktop     Mexico              170   
2  exp_user_2      control          1      mobile   Colombia              140   
3  exp_user_3  tratamiento          0      m

In [ ]:
#Contar conversiones por grupo
conversiones = df.groupby('variante')['convirtio'].sum()
n = df.groupby('variante')['convirtio'].count()

#Z-test de dos proporciones
stat, p_value = proportions_ztest(count=conversiones, nobs=n)

stat, p_value

(-0.8132782986429474, 0.41605851639119995)

**stat = -0.8133**

**p_value = 0.4160**

El **p-value = 0.416** es MUY mayor que 0.05.

Se aplicó un test de dos proporciones para comparar la tasa de conversión entre el grupo control y el grupo con la nueva UI. El **valor p** obtenido fue 0.416, mayor al nivel de significanca de 0.05.
Por lo tanto, no existe evidencia estadística suficiente para afirmar que la nueva UI del checkout haya generado un impacto significativo en la conversión.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Se cargan los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

El proyecto puede consultarse mediante:

🔗 Imágenes al final de este repositorio

✔️ Archivo incluido directamente en este repositorio
